In [0]:
%pip install openmeteo-requests requests-cache retry-requests geopy pandas
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 691.7/691.7 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.2 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-470a314f-cf62-44ad-98d8-f358182781db
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
  Attempting uninstall: attrs
    Found existing installation: attrs 24.3.0
    Not uninstalling attrs at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-470a314f-cf62-44ad-98d8-f358182781db
    Can't uninstall 'attrs'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from pyspark.sql import functions as F
from datetime import datetime
import time

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [0]:
# Get unique counties from rma_county_data_clean using PySpark DataFrame API
rma_df = spark.table("`db_hackathon_2-28`.`default`.`rma_county_data_clean`")

# Select distinct counties and trim whitespace from all string columns
counties_df = rma_df.select(
    F.trim(F.col("State Name")).alias("state_name"),
    F.trim(F.col("State Abbreviation")).alias("state_abbr"),
    F.trim(F.col("County Name")).alias("county_name")
).filter(
    (F.col("County Name").isNotNull()) & 
    (F.col("State Name").isNotNull())
).distinct().orderBy("state_name", "county_name")

# Convert to Pandas for easier geocoding
counties_pd = counties_df.toPandas()

print(f"Found {len(counties_pd)} unique counties to fetch forecasts for")
print("\nSample counties:")
display(counties_pd.head(10))

Found 1312 unique counties to fetch forecasts for

Sample counties:


state_name,state_abbr,county_name
Alabama,AL,Baldwin
Alabama,AL,Cherokee
Alabama,AL,Colbert
Alabama,AL,DeKalb
Alabama,AL,Jackson
Alabama,AL,Lauderdale
Alabama,AL,Lawrence
Alabama,AL,Limestone
Alabama,AL,Madison
Alabama,AL,Marshall


In [0]:
# Load pre-geocoded counties from the geocoded_counties table
# This table was created by the geocode_counties_one_time notebook

print("Loading pre-geocoded counties from table...")

geocode_table = "`db_hackathon_2-28`.`default`.`geocoded_counties`"
counties_spark_df = spark.table(geocode_table)

# Convert to Pandas for easier iteration in the forecast fetch loop
counties_with_coords = counties_spark_df.toPandas()

print(f"✓ Loaded {len(counties_with_coords)} pre-geocoded counties from {geocode_table}")
print(f"\nSample geocoded counties:")
display(counties_with_coords.head(10))

Loading pre-geocoded counties from table...
✓ Loaded 1311 pre-geocoded counties from `db_hackathon_2-28`.`default`.`geocoded_counties`

Sample geocoded counties:


state_name,state_abbr,county_name,latitude,longitude
Missouri,MO,Shelby,39.8004535,-92.071021
Missouri,MO,Stoddard,36.8556646,-89.9451636
Missouri,MO,Sullivan,40.1993002,-93.1140993
Missouri,MO,Vernon,37.8639924,-94.3800151
Missouri,MO,Warren,38.7535106,-91.1453948
Missouri,MO,Worth,40.4684837,-94.4176406
Nebraska,NE,Adams,40.5125798,-98.5148588
Nebraska,NE,Antelope,42.1864809,-98.0716943
Nebraska,NE,Banner,41.5584661,-103.7433155
Nebraska,NE,Boone,41.7103804,-98.0544842


In [0]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://api.open-meteo.com/v1/forecast"

def fetch_county_forecast(row):
    """
    Fetch 16-day weather forecast for a county from Open-Meteo API.
    Returns a list of dictionaries with daily forecast data.
    """
    try:
        params = {
            "latitude": row['latitude'],
            "longitude": row['longitude'],
            "daily": ["temperature_2m_max", "temperature_2m_min", "precipitation_sum", "snowfall_sum"],
            "timezone": "America/New_York",  # Will be auto-adjusted by API
            "forecast_days": 16,
        }
        
        responses = openmeteo.weather_api(url, params=params)
        response = responses[0]
        
        # Process daily data
        daily = response.Daily()
        daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
        daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
        daily_precipitation_sum = daily.Variables(2).ValuesAsNumpy()
        daily_snowfall_sum = daily.Variables(3).ValuesAsNumpy()
        
        # Create date range
        dates = pd.date_range(
            start=pd.to_datetime(daily.Time(), unit="s", utc=True),
            end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=daily.Interval()),
            inclusive="left"
        )
        
        # Create list of records
        forecast_records = []
        for i, date in enumerate(dates):
            forecast_records.append({
                'state_name': row['state_name'],
                'state_abbr': row['state_abbr'],
                'county_name': row['county_name'],
                'latitude': row['latitude'],
                'longitude': row['longitude'],
                'forecast_date': date.date(),
                'temperature_2m_max': float(daily_temperature_2m_max[i]),
                'temperature_2m_min': float(daily_temperature_2m_min[i]),
                'precipitation_sum': float(daily_precipitation_sum[i]),
                'snowfall_sum': float(daily_snowfall_sum[i]),
                'fetched_at': datetime.now()
            })
        
        return forecast_records
    
    except Exception as e:
        print(f"Error fetching forecast for {row['county_name']}, {row['state_name']}: {e}")
        return []

print("Fetching weather forecasts from Open-Meteo API...")
print(f"This will fetch 16-day forecasts for {len(counties_with_coords)} counties\n")

# Fetch forecasts for all counties
all_forecasts = []
for idx, row in counties_with_coords.iterrows():
    if idx % 10 == 0:
        print(f"Progress: {idx}/{len(counties_with_coords)} counties processed...")
    
    county_forecasts = fetch_county_forecast(row)
    all_forecasts.extend(county_forecasts)
    
    # Small delay to respect API rate limits (Open-Meteo allows 10k calls/day)
    time.sleep(0.1)

print(f"\n\u2713 Successfully fetched {len(all_forecasts)} forecast records ({len(all_forecasts)//16} counties × 16 days)")

# Convert to DataFrame
forecasts_df = pd.DataFrame(all_forecasts)

print(f"\nForecast data shape: {forecasts_df.shape}")
print("\nSample forecast data:")
display(forecasts_df.head(10))

Fetching weather forecasts from Open-Meteo API...
This will fetch 16-day forecasts for 1311 counties

Progress: 0/1311 counties processed...
Progress: 10/1311 counties processed...
Progress: 20/1311 counties processed...
Progress: 30/1311 counties processed...
Progress: 40/1311 counties processed...
Progress: 50/1311 counties processed...
Progress: 60/1311 counties processed...
Progress: 70/1311 counties processed...
Progress: 80/1311 counties processed...
Progress: 90/1311 counties processed...
Progress: 100/1311 counties processed...
Progress: 110/1311 counties processed...
Progress: 120/1311 counties processed...
Progress: 130/1311 counties processed...
Progress: 140/1311 counties processed...
Progress: 150/1311 counties processed...
Progress: 160/1311 counties processed...
Progress: 170/1311 counties processed...
Progress: 180/1311 counties processed...
Progress: 190/1311 counties processed...
Progress: 200/1311 counties processed...
Progress: 210/1311 counties processed...
Progres

state_name,state_abbr,county_name,latitude,longitude,forecast_date,temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,fetched_at
Alabama,AL,Baldwin,30.5677527,-87.7324392,2026-02-28,23.569499969482422,10.519499778747559,0.0,0.0,2026-02-28T20:23:00.972Z
Alabama,AL,Baldwin,30.5677527,-87.7324392,2026-03-01,23.91950035095215,10.619500160217285,0.0,0.0,2026-02-28T20:23:00.972Z
Alabama,AL,Baldwin,30.5677527,-87.7324392,2026-03-02,23.37200164794922,9.819499969482422,0.0,0.0,2026-02-28T20:23:00.972Z
Alabama,AL,Baldwin,30.5677527,-87.7324392,2026-03-03,22.972000122070312,12.871999740600586,0.0,0.0,2026-02-28T20:23:00.972Z
Alabama,AL,Baldwin,30.5677527,-87.7324392,2026-03-04,24.57200050354004,15.772000312805176,0.800000011920929,0.0,2026-02-28T20:23:00.973Z
Alabama,AL,Baldwin,30.5677527,-87.7324392,2026-03-05,26.172000885009766,17.272001266479492,0.0,0.0,2026-02-28T20:23:00.973Z
Alabama,AL,Baldwin,30.5677527,-87.7324392,2026-03-06,26.12200164794922,17.722000122070312,0.0,0.0,2026-02-28T20:23:00.973Z
Alabama,AL,Baldwin,30.5677527,-87.7324392,2026-03-07,26.32200050354004,17.972000122070312,0.0,0.0,2026-02-28T20:23:00.973Z
Alabama,AL,Baldwin,30.5677527,-87.7324392,2026-03-08,26.722000122070312,17.272001266479492,0.0,0.0,2026-02-28T20:23:00.973Z
Alabama,AL,Baldwin,30.5677527,-87.7324392,2026-03-09,26.922000885009766,16.272001266479492,0.0,0.0,2026-02-28T20:23:00.973Z


In [0]:
# Convert Pandas DataFrame to Spark DataFrame
forecasts_spark_df = spark.createDataFrame(forecasts_df)

# Define table name
table_name = "`db_hackathon_2-28`.`default`.`county_weather_forecasts_16day`"

# Write to Delta table (overwrite mode - replace existing data)
forecasts_spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"\u2713 Successfully saved forecast data to table: {table_name}")
print(f"\nTable statistics:")
print(f"  - Total rows: {forecasts_spark_df.count():,}")
print(f"  - Unique counties: {forecasts_spark_df.select('county_name').distinct().count()}")
print(f"  - Date range: {forecasts_spark_df.agg(F.min('forecast_date')).collect()[0][0]} to {forecasts_spark_df.agg(F.max('forecast_date')).collect()[0][0]}")
print(f"  - Columns: {', '.join(forecasts_spark_df.columns)}")

✓ Successfully saved forecast data to table: `db_hackathon_2-28`.`default`.`county_weather_forecasts_16day`

Table statistics:
  - Total rows: 20,976
  - Unique counties: 829
  - Date range: 2026-02-28 to 2026-03-15
  - Columns: state_name, state_abbr, county_name, latitude, longitude, forecast_date, temperature_2m_max, temperature_2m_min, precipitation_sum, snowfall_sum, fetched_at


In [0]:
# Query the table to verify
result_df = spark.sql(f"""
    SELECT 
        state_name,
        county_name,
        forecast_date,
        temperature_2m_max,
        temperature_2m_min,
        precipitation_sum,
        snowfall_sum
    FROM {table_name}
    ORDER BY state_name, county_name, forecast_date
    LIMIT 20
""")

print("Sample data from the new table:")
display(result_df)

Sample data from the new table:


state_name,county_name,forecast_date,temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum
Alabama,Baldwin,2026-02-28,23.569499969482422,10.519499778747559,0.0,0.0
Alabama,Baldwin,2026-03-01,23.91950035095215,10.619500160217285,0.0,0.0
Alabama,Baldwin,2026-03-02,23.37200164794922,9.819499969482422,0.0,0.0
Alabama,Baldwin,2026-03-03,22.972000122070312,12.871999740600586,0.0,0.0
Alabama,Baldwin,2026-03-04,24.57200050354004,15.772000312805176,0.800000011920929,0.0
Alabama,Baldwin,2026-03-05,26.172000885009766,17.272001266479492,0.0,0.0
Alabama,Baldwin,2026-03-06,26.12200164794922,17.722000122070312,0.0,0.0
Alabama,Baldwin,2026-03-07,26.32200050354004,17.972000122070312,0.0,0.0
Alabama,Baldwin,2026-03-08,26.722000122070312,17.272001266479492,0.0,0.0
Alabama,Baldwin,2026-03-09,26.922000885009766,16.272001266479492,0.0,0.0


In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from datetime import timedelta, date

print("Building 90-day extended forecast by blending 16-day forecasts with historical climatology...\n")

# Step 1: Load the 16-day forecast data and trim all string columns
forecast_16day = spark.table("`db_hackathon_2-28`.`default`.`county_weather_forecasts_16day`") \
    .withColumn("state_name", F.trim(F.col("state_name"))) \
    .withColumn("state_abbr", F.trim(F.col("state_abbr"))) \
    .withColumn("county_name", F.trim(F.col("county_name")))

# Step 2: Get historical climatological averages from NOAA data
# We'll calculate 30-year averages for each day of year and location
print("Calculating climatological averages from NOAA historical data...")

noaa_df = spark.table("rearc_daily_weather_observations_noaa.esg_noaa_ghcn.noaa_ghcn_daily")

# Extract state from station name (last 2 characters) and calculate daily climatology
climatology = noaa_df.filter(
    (F.col("date") >= "1990-01-01") & 
    (F.col("date") <= "2025-12-31") &
    (F.col("temp_max").isNotNull()) &
    (F.col("temp_min").isNotNull())
).withColumn(
    "state_abbr", 
    F.trim(F.substring(F.col("name"), -2, 2))
).withColumn(
    "month", F.month("date")
).withColumn(
    "day", F.dayofmonth("date")
).groupBy("state_abbr", "month", "day").agg(
    (F.avg("temp_max") / 10.0).alias("clim_temp_max"),
    (F.avg("temp_min") / 10.0).alias("clim_temp_min"),
    (F.avg("precipitation") / 10.0).alias("clim_precipitation")
)

print(f"✓ Calculated climatology for {climatology.count():,} day-of-year/state combinations\n")

# Step 3: Generate 90-day date range starting from today
today = date.today()
date_range = [(today + timedelta(days=i),) for i in range(90)]
date_df = spark.createDataFrame(date_range, ["forecast_date"]) \
    .withColumn("month", F.month("forecast_date")) \
    .withColumn("day", F.dayofmonth("forecast_date")) \
    .withColumn("days_out", F.datediff("forecast_date", F.lit(today)))

print(f"Generated 90-day date range from {today} to {today + timedelta(days=89)}\n")

Building 90-day extended forecast by blending 16-day forecasts with historical climatology...

Calculating climatological averages from NOAA historical data...
✓ Calculated climatology for 74,301 day-of-year/state combinations

Generated 90-day date range from 2026-02-28 to 2026-05-28



In [0]:
# Step 4: Get unique counties with their coordinates
counties = forecast_16day.select(
    "state_name", "state_abbr", "county_name", "latitude", "longitude"
).distinct()

print(f"Processing {counties.count()} counties...\n")

# Step 5: Create cross join of counties and 90-day date range
extended_base = counties.crossJoin(date_df)

# Step 6: Left join with actual 16-day forecasts
extended_with_forecast = extended_base.join(
    forecast_16day.select(
        "state_abbr", "county_name", "forecast_date",
        F.col("temperature_2m_max").alias("forecast_temp_max"),
        F.col("temperature_2m_min").alias("forecast_temp_min"),
        F.col("precipitation_sum").alias("forecast_precipitation"),
        F.col("snowfall_sum").alias("forecast_snowfall")
    ),
    ["state_abbr", "county_name", "forecast_date"],
    "left"
)

# Step 7: Join with climatological averages
extended_with_clim = extended_with_forecast.join(
    climatology,
    ["state_abbr", "month", "day"],
    "left"
)

# Step 8: Create blended forecast
# - Days 1-16: Use actual forecast
# - Days 17-30: Blend forecast and climatology (70% clim, 30% last forecast value)
# - Days 31-90: Use pure climatology

extended_forecast = extended_with_clim.withColumn(
    "last_forecast_temp_max",
    F.last("forecast_temp_max", ignorenulls=True).over(
        Window.partitionBy("state_abbr", "county_name").orderBy("forecast_date")
    )
).withColumn(
    "last_forecast_temp_min",
    F.last("forecast_temp_min", ignorenulls=True).over(
        Window.partitionBy("state_abbr", "county_name").orderBy("forecast_date")
    )
).withColumn(
    "blend_weight",
    F.when(F.col("days_out") <= 16, 0.0)  # Days 1-16: 0% climatology (100% forecast)
     .when(F.col("days_out") <= 30, (F.col("days_out") - 16) / 14.0)  # Days 17-30: gradual blend
     .otherwise(1.0)  # Days 31-90: 100% climatology
).withColumn(
    "temperature_max",
    F.coalesce(
        F.col("forecast_temp_max"),  # Use forecast if available
        (1 - F.col("blend_weight")) * F.col("last_forecast_temp_max") + 
        F.col("blend_weight") * F.col("clim_temp_max")  # Blend for days 17+
    )
).withColumn(
    "temperature_min",
    F.coalesce(
        F.col("forecast_temp_min"),
        (1 - F.col("blend_weight")) * F.col("last_forecast_temp_min") + 
        F.col("blend_weight") * F.col("clim_temp_min")
    )
).withColumn(
    "precipitation",
    F.coalesce(
        F.col("forecast_precipitation"),
        F.col("clim_precipitation")
    )
).withColumn(
    "snowfall",
    F.coalesce(
        F.col("forecast_snowfall"),
        F.lit(0.0)  # Use 0 for snowfall in climatology (not in NOAA daily)
    )
).withColumn(
    "forecast_type",
    F.when(F.col("days_out") <= 16, "forecast")
     .when(F.col("days_out") <= 30, "blended")
     .otherwise("climatology")
).withColumn(
    "uncertainty_range_celsius",
    # Uncertainty increases linearly from ±1°C (day 1) to ±5°C (day 90)
    1.0 + (F.col("days_out") * 4.0 / 89.0)
).withColumn(
    "created_at",
    F.current_timestamp()
)

# Select final columns
final_forecast = extended_forecast.select(
    "state_name",
    "state_abbr",
    "county_name",
    "latitude",
    "longitude",
    "forecast_date",
    "days_out",
    "temperature_max",
    "temperature_min",
    "precipitation",
    "snowfall",
    "forecast_type",
    "uncertainty_range_celsius",
    "created_at"
).orderBy("state_name", "county_name", "forecast_date")

print("✓ Extended forecast created with blended methodology\n")

Processing 1311 counties...

✓ Extended forecast created with blended methodology



In [0]:
# Save to new Delta table
table_name_90day = "`db_hackathon_2-28`.`default`.`county_weather_forecasts_90day`"

final_forecast.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name_90day)

print(f"✓ Successfully saved 90-day extended forecast to: {table_name_90day}\n")

# Display statistics
stats = final_forecast.groupBy("forecast_type").agg(
    F.count("*").alias("record_count"),
    F.avg("temperature_max").alias("avg_temp_max"),
    F.avg("uncertainty_range_celsius").alias("avg_uncertainty")
)

print("Forecast breakdown by type:")
stats.show()

print(f"\nTotal records: {final_forecast.count():,}")
print(f"Unique counties: {final_forecast.select('county_name').distinct().count()}")
print(f"Date range: {final_forecast.agg(F.min('forecast_date')).collect()[0][0]} to {final_forecast.agg(F.max('forecast_date')).collect()[0][0]}")
print(f"\nColumns: {', '.join(final_forecast.columns)}")

✓ Successfully saved 90-day extended forecast to: `db_hackathon_2-28`.`default`.`county_weather_forecasts_90day`

Forecast breakdown by type:
+-------------+------------+------------------+------------------+
|forecast_type|record_count|      avg_temp_max|   avg_uncertainty|
+-------------+------------+------------------+------------------+
|     forecast|       22287| 12.86965678844837|1.3595505617978325|
|      blended|       18354|18.506794639403193|2.0561797752809383|
|  climatology|       77349|26.778438131601405| 3.696629213483263|
+-------------+------------+------------------+------------------+


Total records: 117,990
Unique counties: 828
Date range: 2026-02-28 to 2026-05-28

Columns: state_name, state_abbr, county_name, latitude, longitude, forecast_date, days_out, temperature_max, temperature_min, precipitation, snowfall, forecast_type, uncertainty_range_celsius, created_at


In [0]:
# Show sample data for one county across different forecast periods
sample_county = final_forecast.filter(
    (F.col("state_abbr") == " AL") & 
    (F.col("county_name") == " Baldwin")
).select(
    "forecast_date",
    "days_out",
    "temperature_max",
    "temperature_min",
    "precipitation",
    "forecast_type",
    "uncertainty_range_celsius"
)

print("Sample 90-day forecast for Baldwin County, Alabama:\n")
print("First 16 days (actual forecast):")
sample_county.filter(F.col("days_out") <= 16).show(16, truncate=False)

print("\nDays 17-30 (blended forecast/climatology):")
sample_county.filter((F.col("days_out") > 16) & (F.col("days_out") <= 30)).show(14, truncate=False)

print("\nDays 60-75 (climatology-based):")
sample_county.filter((F.col("days_out") >= 60) & (F.col("days_out") < 75)).limit(15).show(truncate=False)

Sample 90-day forecast for Baldwin County, Alabama:

First 16 days (actual forecast):
+-------------+--------+------------------+------------------+------------------+-------------+-------------------------+
|forecast_date|days_out|temperature_max   |temperature_min   |precipitation     |forecast_type|uncertainty_range_celsius|
+-------------+--------+------------------+------------------+------------------+-------------+-------------------------+
|2026-02-28   |0       |23.569499969482422|10.519499778747559|0.0               |forecast     |1.0                      |
|2026-03-01   |1       |23.91950035095215 |10.619500160217285|0.0               |forecast     |1.0449438202247192       |
|2026-03-02   |2       |23.37200164794922 |9.819499969482422 |0.0               |forecast     |1.0898876404494382       |
|2026-03-03   |3       |22.972000122070312|12.871999740600586|0.0               |forecast     |1.1348314606741572       |
|2026-03-04   |4       |24.57200050354004 |15.77200031280517

In [0]:
# Check what's in the table for a sample county
diagnostic = spark.sql("""
    SELECT 
        forecast_date,
        days_out,
        temperature_max,
        temperature_min,
        precipitation,
        forecast_type,
        state_abbr,
        county_name
    FROM `db_hackathon_2-28`.`default`.`county_weather_forecasts_90day`
    WHERE state_abbr = ' AL' AND county_name = ' Baldwin'
    ORDER BY forecast_date
    LIMIT 30
""")

print("Checking actual data in the table:")
diagnostic.show(30, truncate=False)

# Check if climatology data exists for AL
print("\nChecking climatology data for Alabama:")
spark.sql("""
    SELECT state_abbr, month, day, clim_temp_max, clim_temp_min, clim_precipitation
    FROM (
        SELECT 
            SUBSTRING(name, -2, 2) as state_abbr,
            MONTH(date) as month,
            DAYOFMONTH(date) as day,
            AVG(temp_max) / 10.0 as clim_temp_max,
            AVG(temp_min) / 10.0 as clim_temp_min,
            AVG(precipitation) / 10.0 as clim_precipitation
        FROM rearc_daily_weather_observations_noaa.esg_noaa_ghcn.noaa_ghcn_daily
        WHERE date >= '1990-01-01' 
            AND date <= '2025-12-31'
            AND temp_max IS NOT NULL
            AND SUBSTRING(name, -2, 2) = 'AL'
        GROUP BY SUBSTRING(name, -2, 2), MONTH(date), DAYOFMONTH(date)
    )
    WHERE month = 3 AND day BETWEEN 17 AND 20
    ORDER BY month, day
""").show(truncate=False)

Checking actual data in the table:
+-------------+--------+------------------+------------------+------------------+-------------+----------+-----------+
|forecast_date|days_out|temperature_max   |temperature_min   |precipitation     |forecast_type|state_abbr|county_name|
+-------------+--------+------------------+------------------+------------------+-------------+----------+-----------+
|2026-02-28   |0       |23.569499969482422|10.519499778747559|0.0               |forecast     | AL       | Baldwin   |
|2026-03-01   |1       |23.91950035095215 |10.619500160217285|0.0               |forecast     | AL       | Baldwin   |
|2026-03-02   |2       |23.37200164794922 |9.819499969482422 |0.0               |forecast     | AL       | Baldwin   |
|2026-03-03   |3       |22.972000122070312|12.871999740600586|0.0               |forecast     | AL       | Baldwin   |
|2026-03-04   |4       |24.57200050354004 |15.772000312805176|0.800000011920929 |forecast     | AL       | Baldwin   |
|2026-03-05  

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from datetime import timedelta, date

print("Building simplified 90-day forecast with clean transition from forecast to climatology...\n")

# Step 1: Load the 16-day forecast data and trim all string columns
forecast_16day = spark.table("`db_hackathon_2-28`.`default`.`county_weather_forecasts_16day`") \
    .withColumn("state_name", F.trim(F.col("state_name"))) \
    .withColumn("state_abbr", F.trim(F.col("state_abbr"))) \
    .withColumn("county_name", F.trim(F.col("county_name")))

# Step 2: Calculate climatology by state, month, and day
print("Calculating climatological averages from NOAA historical data...")

noaa_df = spark.table("rearc_daily_weather_observations_noaa.esg_noaa_ghcn.noaa_ghcn_daily")

climatology = noaa_df.filter(
    (F.col("date") >= "1990-01-01") & 
    (F.col("date") <= "2025-12-31") &
    (F.col("temp_max").isNotNull()) &
    (F.col("temp_min").isNotNull())
).withColumn(
    "state_abbr", 
    F.trim(F.substring(F.col("name"), -2, 2))
).withColumn(
    "month", F.month("date")
).withColumn(
    "day", F.dayofmonth("date")
).groupBy("state_abbr", "month", "day").agg(
    (F.avg("temp_max") / 10.0).alias("clim_temp_max"),
    (F.avg("temp_min") / 10.0).alias("clim_temp_min"),
    (F.avg("precipitation") / 10.0).alias("clim_precipitation")
)

print(f"✓ Calculated climatology for {climatology.count():,} day-of-year/state combinations\n")

# Step 3: Generate 90-day date range
today = date.today()
date_range = [(today + timedelta(days=i),) for i in range(90)]
date_df = spark.createDataFrame(date_range, ["forecast_date"]) \
    .withColumn("month", F.month("forecast_date")) \
    .withColumn("day", F.dayofmonth("forecast_date")) \
    .withColumn("days_out", F.datediff("forecast_date", F.lit(today)))

print(f"Generated 90-day date range from {today} to {today + timedelta(days=89)}\n")

Building simplified 90-day forecast with clean transition from forecast to climatology...

Calculating climatological averages from NOAA historical data...
✓ Calculated climatology for 74,301 day-of-year/state combinations

Generated 90-day date range from 2026-02-28 to 2026-05-28



In [0]:
# Step 4: Get unique counties with coordinates
counties = forecast_16day.select(
    "state_name", "state_abbr", "county_name", "latitude", "longitude"
).distinct()

print(f"Processing {counties.count()} counties...\n")

# Step 5: Cross join counties with 90-day date range
extended_base = counties.crossJoin(date_df)

# Step 6: Left join with 16-day forecasts
extended_with_forecast = extended_base.join(
    forecast_16day.select(
        "state_abbr", "county_name", "forecast_date",
        F.col("temperature_2m_max").alias("forecast_temp_max"),
        F.col("temperature_2m_min").alias("forecast_temp_min"),
        F.col("precipitation_sum").alias("forecast_precipitation"),
        F.col("snowfall_sum").alias("forecast_snowfall")
    ),
    ["state_abbr", "county_name", "forecast_date"],
    "left"
)

# Step 7: Join with climatology
extended_with_clim = extended_with_forecast.join(
    climatology,
    ["state_abbr", "month", "day"],
    "left"
)

# Step 8: Get day 16 forecast values for blending days 17-20
day_16_values = extended_with_clim.filter(F.col("days_out") == 15) \
    .select(
        "state_abbr", "county_name",
        F.col("forecast_temp_max").alias("day16_temp_max"),
        F.col("forecast_temp_min").alias("day16_temp_min"),
        F.col("forecast_precipitation").alias("day16_precipitation")
    )

extended_with_day16 = extended_with_clim.join(
    day_16_values,
    ["state_abbr", "county_name"],
    "left"
)

print("✓ Joined forecast, climatology, and day 16 values\n")

Processing 1311 counties...

✓ Joined forecast, climatology, and day 16 values



In [0]:
# Step 9: Apply simplified blending logic
# - Days 0-15 (days_out 0-15): Use actual 16-day forecast
# - Days 16-19 (days_out 16-19): Average of day 16 forecast and climatology
# - Days 20-89 (days_out 20-89): Use pure climatology

simplified_forecast = extended_with_day16.withColumn(
    "temperature_max",
    F.when(
        F.col("days_out") <= 15,
        F.col("forecast_temp_max")  # Days 0-15: actual forecast
    ).when(
        (F.col("days_out") >= 16) & (F.col("days_out") <= 19),
        F.coalesce(
            (F.col("day16_temp_max") + F.col("clim_temp_max")) / 2.0,  # Days 16-19: average if both exist
            F.col("day16_temp_max"),  # Fall back to day 16 if no climatology
            F.col("clim_temp_max")  # Fall back to climatology if no day 16
        )
    ).otherwise(
        F.coalesce(
            F.col("clim_temp_max"),  # Days 20+: climatology
            F.col("day16_temp_max")  # Fallback to day 16 if no climatology
        )
    )
).withColumn(
    "temperature_min",
    F.when(
        F.col("days_out") <= 15,
        F.col("forecast_temp_min")
    ).when(
        (F.col("days_out") >= 16) & (F.col("days_out") <= 19),
        F.coalesce(
            (F.col("day16_temp_min") + F.col("clim_temp_min")) / 2.0,
            F.col("day16_temp_min"),
            F.col("clim_temp_min")
        )
    ).otherwise(
        F.coalesce(
            F.col("clim_temp_min"),
            F.col("day16_temp_min")
        )
    )
).withColumn(
    "precipitation",
    F.when(
        F.col("days_out") <= 15,
        F.col("forecast_precipitation")
    ).when(
        (F.col("days_out") >= 16) & (F.col("days_out") <= 19),
        F.coalesce(
            (F.coalesce(F.col("day16_precipitation"), F.lit(0.0)) + F.col("clim_precipitation")) / 2.0,
            F.col("day16_precipitation"),
            F.col("clim_precipitation"),
            F.lit(0.0)
        )
    ).otherwise(
        F.coalesce(
            F.col("clim_precipitation"),
            F.lit(0.0)
        )
    )
).withColumn(
    "snowfall",
    F.when(
        F.col("days_out") <= 15,
        F.col("forecast_snowfall")
    ).otherwise(
        F.lit(0.0)  # No snowfall data in climatology
    )
).withColumn(
    "forecast_type",
    F.when(F.col("days_out") <= 15, "forecast")
     .when((F.col("days_out") >= 16) & (F.col("days_out") <= 19), "transition")
     .otherwise("climatology")
).withColumn(
    "uncertainty_range_celsius",
    # Uncertainty: ±1°C for days 0-15, ±2°C for days 16-19, ±3-5°C for climatology
    F.when(F.col("days_out") <= 15, 1.0)
     .when((F.col("days_out") >= 16) & (F.col("days_out") <= 19), 2.0)
     .otherwise(3.0 + (F.col("days_out") - 20) * 2.0 / 70.0)  # 3-5°C for days 20-89
).withColumn(
    "created_at",
    F.current_timestamp()
)

# Select final columns
final_simplified_forecast = simplified_forecast.select(
    "state_name",
    "state_abbr",
    "county_name",
    "latitude",
    "longitude",
    "forecast_date",
    "days_out",
    "temperature_max",
    "temperature_min",
    "precipitation",
    "snowfall",
    "forecast_type",
    "uncertainty_range_celsius",
    "created_at"
).orderBy("state_name", "county_name", "forecast_date")

print("✓ Simplified forecast created with clean transitions\n")

✓ Simplified forecast created with clean transitions



In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from datetime import timedelta, date
import math

print("Building 90-day forecast with COUNTY-LEVEL climatology...\n")

# Step 1: Load the 16-day forecast data and trim all string columns
forecast_16day = spark.table("`db_hackathon_2-28`.`default`.`county_weather_forecasts_16day`") \
    .withColumn("state_name", F.trim(F.col("state_name"))) \
    .withColumn("state_abbr", F.trim(F.col("state_abbr"))) \
    .withColumn("county_name", F.trim(F.col("county_name")))

# Get unique counties with coordinates
counties = forecast_16day.select(
    "state_name", "state_abbr", "county_name", "latitude", "longitude"
).distinct()

print(f"Processing {counties.count()} counties...\n")

# Step 2: Get all NOAA weather stations with their locations
print("Loading NOAA weather stations...")
noaa_stations = spark.sql("""
    SELECT DISTINCT
        station,
        latitude,
        longitude,
        name
    FROM rearc_daily_weather_observations_noaa.esg_noaa_ghcn.noaa_ghcn_daily
    WHERE date >= '1990-01-01' 
        AND date <= '2025-12-31'
        AND temp_max IS NOT NULL
        AND latitude IS NOT NULL
        AND longitude IS NOT NULL
""")

print(f"✓ Found {noaa_stations.count():,} weather stations\n")

# Step 3: Create a cross join to find distances from each county to each station
print("Calculating distances from counties to weather stations...")

# Use Haversine formula to calculate distance
def haversine_distance_udf():
    def haversine(lat1, lon1, lat2, lon2):
        if any(v is None for v in [lat1, lon1, lat2, lon2]):
            return None
        
        # Convert to radians
        lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
        
        # Haversine formula
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
        c = 2 * math.asin(math.sqrt(a))
        
        # Radius of Earth in kilometers
        r = 6371
        return r * c
    
    return F.udf(haversine, "double")

haversine_udf = haversine_distance_udf()

# Cross join counties with stations and calculate distances
county_station_distances = counties.crossJoin(
    noaa_stations.select(
        F.col("station"),
        F.col("latitude").alias("station_lat"),
        F.col("longitude").alias("station_lon")
    )
).withColumn(
    "distance_km",
    haversine_udf(
        F.col("latitude"), F.col("longitude"),
        F.col("station_lat"), F.col("station_lon")
    )
)

print("✓ Distance calculations complete\n")

# Step 4: For each county, find the 5 nearest stations
print("Finding 5 nearest stations for each county...")

nearest_stations = county_station_distances.withColumn(
    "rank",
    F.row_number().over(
        Window.partitionBy("state_abbr", "county_name")
              .orderBy(F.col("distance_km").asc_nulls_last())
    )
).filter(F.col("rank") <= 5).select(
    "state_abbr",
    "county_name",
    "station",
    "distance_km"
)

print(f"✓ Found nearest stations for {nearest_stations.select('state_abbr', 'county_name').distinct().count()} counties\n")

# Show sample
print("Sample: Nearest stations for Edgar County, IL:")
nearest_stations.filter(
    (F.col("state_abbr") == "IL") & (F.col("county_name") == "Edgar")
).show(truncate=False)

Building 90-day forecast with COUNTY-LEVEL climatology...

Processing 1311 counties...

Loading NOAA weather stations...
✓ Found 20,349 weather stations

Calculating distances from counties to weather stations...
✓ Distance calculations complete

Finding 5 nearest stations for each county...
✓ Found nearest stations for 1311 counties

Sample: Nearest stations for Edgar County, IL:
+----------+-----------+-----------+------------------+
|state_abbr|county_name|station    |distance_km       |
+----------+-----------+-----------+------------------+
|IL        |Edgar      |USC00111636|15.830609436148416|
|IL        |Edgar      |USC00117952|33.7977297206038  |
|IL        |Edgar      |USW00003868|43.51445961764834 |
|IL        |Edgar      |USW00093823|44.62066648268822 |
|IL        |Edgar      |USC00118684|49.56186714823181 |
+----------+-----------+-----------+------------------+



In [0]:
# Step 5: Calculate climatology using only nearby stations for each county
print("Calculating county-specific climatology from nearby stations...\n")

# Join nearest stations with full NOAA data
noaa_full = spark.table("reac_daily_weather_observations_noaa.esg_noaa_ghcn.noaa_ghcn_daily")

county_climatology = nearest_stations.join(
    noaa_full.filter(
        (F.col("date") >= "1990-01-01") & 
        (F.col("date") <= "2025-12-31") &
        (F.col("temp_max").isNotNull()) &
        (F.col("temp_min").isNotNull())
    ).select(
        "station",
        "date",
        "temp_max",
        "temp_min",
        "precipitation"
    ),
    "station",
    "inner"
).withColumn(
    "month", F.month("date")
).withColumn(
    "day", F.dayofmonth("date")
).groupBy("state_abbr", "county_name", "month", "day").agg(
    (F.avg("temp_max") / 10.0).alias("clim_temp_max"),
    (F.avg("temp_min") / 10.0).alias("clim_temp_min"),
    (F.avg("precipitation") / 10.0).alias("clim_precipitation"),
    F.count("*").alias("num_observations")
)

print(f"✓ Calculated county-level climatology for {county_climatology.select('state_abbr', 'county_name').distinct().count()} counties\n")

# Show comparison for Edgar and Vermillion counties on March 18
print("County-level climatology for Edgar (IL) and Vermillion (IN) on March 18:")
county_climatology.filter(
    ((F.col("state_abbr") == "IL") & (F.col("county_name") == "Edgar")) |
    ((F.col("state_abbr") == "IN") & (F.col("county_name") == "Vermillion"))
).filter(
    (F.col("month") == 3) & (F.col("day") == 18)
).show(truncate=False)

print("These values should now be much more similar since the counties are neighbors!")

Calculating county-specific climatology from nearby stations...

✓ Calculated county-level climatology for 1311 counties

County-level climatology for Edgar (IL) and Vermillion (IN) on March 18:
+----------+-----------+-----+---+------------------+------------------+------------------+----------------+
|state_abbr|county_name|month|day|clim_temp_max     |clim_temp_min     |clim_precipitation|num_observations|
+----------+-----------+-----+---+------------------+------------------+------------------+----------------+
|IL        |Edgar      |3    |18 |12.651807228915661|1.2493975903614458|3.2707317073170734|83              |
|IN        |Vermillion |3    |18 |12.792063492063493|1.3873015873015873|3.2403225806451617|63              |
+----------+-----------+-----+---+------------------+------------------+------------------+----------------+

These values should now be much more similar since the counties are neighbors!


In [0]:
# Step 6: Generate 90-day date range
today = date.today()
date_range = [(today + timedelta(days=i),) for i in range(90)]
date_df = spark.createDataFrame(date_range, ["forecast_date"]) \
    .withColumn("month", F.month("forecast_date")) \
    .withColumn("day", F.dayofmonth("forecast_date")) \
    .withColumn("days_out", F.datediff("forecast_date", F.lit(today)))

print(f"Generated 90-day date range from {today} to {today + timedelta(days=89)}\n")

# Step 7: Cross join counties with date range
print("Building extended forecast...")
extended_base = counties.crossJoin(date_df)

# Step 8: Join with 16-day forecasts
extended_with_forecast = extended_base.join(
    forecast_16day.select(
        "state_abbr", "county_name", "forecast_date",
        F.col("temperature_2m_max").alias("forecast_temp_max"),
        F.col("temperature_2m_min").alias("forecast_temp_min"),
        F.col("precipitation_sum").alias("forecast_precipitation"),
        F.col("snowfall_sum").alias("forecast_snowfall")
    ),
    ["state_abbr", "county_name", "forecast_date"],
    "left"
)

# Step 9: Join with COUNTY-LEVEL climatology
extended_with_clim = extended_with_forecast.join(
    county_climatology.select(
        "state_abbr", "county_name", "month", "day",
        "clim_temp_max", "clim_temp_min", "clim_precipitation"
    ),
    ["state_abbr", "county_name", "month", "day"],
    "left"
)

# Step 10: Get day 16 values for transition blending
day_16_values = extended_with_clim.filter(F.col("days_out") == 15) \
    .select(
        "state_abbr", "county_name",
        F.col("forecast_temp_max").alias("day16_temp_max"),
        F.col("forecast_temp_min").alias("day16_temp_min"),
        F.col("forecast_precipitation").alias("day16_precipitation")
    )

extended_with_day16 = extended_with_clim.join(
    day_16_values,
    ["state_abbr", "county_name"],
    "left"
)

print("✓ Joined forecast with county-level climatology\n")

Generated 90-day date range from 2026-02-28 to 2026-05-28

Building extended forecast...
✓ Joined forecast with county-level climatology



In [0]:
# Step 11: Apply simplified blending logic
# - Days 0-15: Use actual 16-day forecast
# - Days 16-19: Average of day 16 forecast and LOCAL climatology
# - Days 20-89: Use LOCAL climatology

local_forecast = extended_with_day16.withColumn(
    "temperature_max",
    F.when(
        F.col("days_out") <= 15,
        F.col("forecast_temp_max")  # Days 0-15: actual forecast
    ).when(
        (F.col("days_out") >= 16) & (F.col("days_out") <= 19),
        F.coalesce(
            (F.col("day16_temp_max") + F.col("clim_temp_max")) / 2.0,
            F.col("day16_temp_max"),
            F.col("clim_temp_max")
        )
    ).otherwise(
        F.coalesce(
            F.col("clim_temp_max"),
            F.col("day16_temp_max")
        )
    )
).withColumn(
    "temperature_min",
    F.when(
        F.col("days_out") <= 15,
        F.col("forecast_temp_min")
    ).when(
        (F.col("days_out") >= 16) & (F.col("days_out") <= 19),
        F.coalesce(
            (F.col("day16_temp_min") + F.col("clim_temp_min")) / 2.0,
            F.col("day16_temp_min"),
            F.col("clim_temp_min")
        )
    ).otherwise(
        F.coalesce(
            F.col("clim_temp_min"),
            F.col("day16_temp_min")
        )
    )
).withColumn(
    "precipitation",
    F.when(
        F.col("days_out") <= 15,
        F.col("forecast_precipitation")
    ).when(
        (F.col("days_out") >= 16) & (F.col("days_out") <= 19),
        F.coalesce(
            (F.coalesce(F.col("day16_precipitation"), F.lit(0.0)) + F.col("clim_precipitation")) / 2.0,
            F.col("day16_precipitation"),
            F.col("clim_precipitation"),
            F.lit(0.0)
        )
    ).otherwise(
        F.coalesce(
            F.col("clim_precipitation"),
            F.lit(0.0)
        )
    )
).withColumn(
    "snowfall",
    F.when(
        F.col("days_out") <= 15,
        F.col("forecast_snowfall")
    ).otherwise(
        F.lit(0.0)
    )
).withColumn(
    "forecast_type",
    F.when(F.col("days_out") <= 15, "forecast")
     .when((F.col("days_out") >= 16) & (F.col("days_out") <= 19), "transition")
     .otherwise("climatology")
).withColumn(
    "uncertainty_range_celsius",
    F.when(F.col("days_out") <= 15, 1.0)
     .when((F.col("days_out") >= 16) & (F.col("days_out") <= 19), 2.0)
     .otherwise(3.0 + (F.col("days_out") - 20) * 2.0 / 70.0)
).withColumn(
    "created_at",
    F.current_timestamp()
)

# Select final columns
final_local_forecast = local_forecast.select(
    "state_name",
    "state_abbr",
    "county_name",
    "latitude",
    "longitude",
    "forecast_date",
    "days_out",
    "temperature_max",
    "temperature_min",
    "precipitation",
    "snowfall",
    "forecast_type",
    "uncertainty_range_celsius",
    "created_at"
).orderBy("state_name", "county_name", "forecast_date")

print("✓ County-level forecast created\n")

# Step 12: Save to new table
table_name_local = "`db_hackathon_2-28`.`default`.`county_weather_forecasts_90day_local`"

final_local_forecast.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name_local)

print(f"✓ Successfully saved county-level forecast to: {table_name_local}\n")

# Display statistics
stats = final_local_forecast.groupBy("forecast_type").agg(
    F.count("*").alias("record_count"),
    F.avg("temperature_max").alias("avg_temp_max"),
    F.avg("uncertainty_range_celsius").alias("avg_uncertainty")
)

print("Forecast breakdown by type:")
stats.show()

print(f"\nTotal records: {final_local_forecast.count():,}")
print(f"Unique counties: {final_local_forecast.select('county_name').distinct().count()}")
print(f"Date range: {final_local_forecast.agg(F.min('forecast_date')).collect()[0][0]} to {final_local_forecast.agg(F.max('forecast_date')).collect()[0][0]}")

✓ County-level forecast created

✓ Successfully saved county-level forecast to: `db_hackathon_2-28`.`default`.`county_weather_forecasts_90day_local`

Forecast breakdown by type:
+-------------+------------+------------------+------------------+
|forecast_type|record_count|      avg_temp_max|   avg_uncertainty|
+-------------+------------+------------------+------------------+
|   transition|        5244|10.404022609354726|               2.0|
|     forecast|       20976|12.931941370134425|               1.0|
|  climatology|       91770| 18.97093350294801|3.9857142857142844|
+-------------+------------+------------------+------------------+


Total records: 117,990
Unique counties: 828
Date range: 2026-02-28 to 2026-05-28


In [0]:
# Verify that Edgar (IL) and Vermillion (IN) now have similar temperatures on March 18
print("VERIFICATION: Comparing neighboring counties on March 18, 2026\n")
print("These counties are only ~30 miles apart and should have similar temperatures:\n")

comparison = spark.sql("""
    SELECT 
        state_name,
        state_abbr,
        county_name,
        forecast_date,
        days_out,
        ROUND(temperature_max, 2) as temp_max_c,
        ROUND(temperature_min, 2) as temp_min_c,
        ROUND(precipitation, 2) as precip_mm,
        forecast_type,
        latitude,
        longitude
    FROM `db_hackathon_2-28`.`default`.`county_weather_forecasts_90day_local`
    WHERE (county_name = 'Edgar' OR county_name = 'Vermillion')
        AND forecast_date = '2026-03-18'
    ORDER BY state_name, county_name
""")

comparison.show(truncate=False)

# Calculate the difference
print("\nTemperature difference between neighboring counties:")
edgar = comparison.filter(F.col("county_name") == "Edgar").first()
vermillion = comparison.filter(F.col("county_name") == "Vermillion").first()

if edgar and vermillion:
    temp_diff = abs(edgar['temp_max_c'] - vermillion['temp_max_c'])
    print(f"  Edgar County, IL: {edgar['temp_max_c']}°C")
    print(f"  Vermillion County, IN: {vermillion['temp_max_c']}°C")
    print(f"  Difference: {temp_diff:.2f}°C")
    print(f"\n✓ Much better! With county-level climatology, neighboring counties")
    print(f"  have similar temperatures instead of 14°C difference!")
else:
    print("  Could not find both counties in results")

VERIFICATION: Comparing neighboring counties on March 18, 2026

These counties are only ~30 miles apart and should have similar temperatures:

+----------+----------+-----------+-------------+--------+----------+----------+---------+-------------+----------+-----------+
|state_name|state_abbr|county_name|forecast_date|days_out|temp_max_c|temp_min_c|precip_mm|forecast_type|latitude  |longitude  |
+----------+----------+-----------+-------------+--------+----------+----------+---------+-------------+----------+-----------+
|Illinois  |IL        |Edgar      |2026-03-18   |18      |8.61      |-1.34     |1.64     |transition   |39.6714952|-87.7340364|
|Indiana   |IN        |Vermillion |2026-03-18   |18      |8.36      |-1.49     |1.62     |transition   |39.8763626|-87.4635672|
+----------+----------+-----------+-------------+--------+----------+----------+---------+-------------+----------+-----------+


Temperature difference between neighboring counties:
  Edgar County, IL: 8.61°C
  Vermi

In [0]:
# Save to Delta table
table_name_simplified = "`db_hackathon_2-28`.`default`.`county_weather_forecasts_90day_simple`"

final_simplified_forecast.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name_simplified)

print(f"✓ Successfully saved simplified 90-day forecast to: {table_name_simplified}\n")

# Display statistics
stats = final_simplified_forecast.groupBy("forecast_type").agg(
    F.count("*").alias("record_count"),
    F.avg("temperature_max").alias("avg_temp_max"),
    F.avg("uncertainty_range_celsius").alias("avg_uncertainty")
)

print("Forecast breakdown by type:")
stats.show()

print(f"\nTotal records: {final_simplified_forecast.count():,}")
print(f"Unique counties: {final_simplified_forecast.select('county_name').distinct().count()}")
print(f"Date range: {final_simplified_forecast.agg(F.min('forecast_date')).collect()[0][0]} to {final_simplified_forecast.agg(F.max('forecast_date')).collect()[0][0]}")
print(f"\nColumns: {', '.join(final_simplified_forecast.columns)}\n")

# Show sample for one county
print("Sample forecast for one county (first 30 days):")
final_simplified_forecast.filter(
    F.col("county_name") == final_simplified_forecast.select("county_name").first()[0]
).select(
    "forecast_date", "days_out", "temperature_max", "temperature_min", 
    "precipitation", "forecast_type", "uncertainty_range_celsius"
).limit(30).show(30, truncate=False)

✓ Successfully saved simplified 90-day forecast to: `db_hackathon_2-28`.`default`.`county_weather_forecasts_90day_simple`

Forecast breakdown by type:
+-------------+------------+------------------+-----------------+
|forecast_type|record_count|      avg_temp_max|  avg_uncertainty|
+-------------+------------+------------------+-----------------+
|     forecast|       20976|12.931941370134425|              1.0|
|   transition|        5244|12.579117075797638|              2.0|
|  climatology|       91770| 16.89877794696033|3.985714285714285|
+-------------+------------+------------------+-----------------+


Total records: 117,990
Unique counties: 828
Date range: 2026-02-28 to 2026-05-28

Columns: state_name, state_abbr, county_name, latitude, longitude, forecast_date, days_out, temperature_max, temperature_min, precipitation, snowfall, forecast_type, uncertainty_range_celsius, created_at

Sample forecast for one county (first 30 days):
+-------------+--------+------------------+-------

# County Weather Forecasts - 16 Day Outlook

## Summary

This notebook fetches 16-day weather forecasts from the Open-Meteo API for all counties in the `rma_county_data_clean` table.

### Data Retrieved

**Weather Parameters (Daily):**
* `temperature_2m_max` - Maximum temperature at 2 meters (°F)
* `temperature_2m_min` - Minimum temperature at 2 meters (°F)
* `precipitation_sum` - Total daily precipitation (mm)
* `snowfall_sum` - Total daily snowfall (cm)

**Location Data:**
* State name and abbreviation
* County name
* Latitude/Longitude coordinates

**Time Information:**
* `forecast_date` - The date of the forecast
* `fetched_at` - Timestamp when data was retrieved

### Output Table

Table name: **`db_hackathon_2-28.default.county_weather_forecasts_16day`**

### Example Queries

```sql
-- Get forecast for a specific county
SELECT * 
FROM `db_hackathon_2-28`.`default`.`county_weather_forecasts_16day`
WHERE county_name = 'King County' 
  AND state_abbr = 'WA'
ORDER BY forecast_date;

-- Find counties with highest expected precipitation in next 7 days
SELECT 
    state_name,
    county_name,
    SUM(precipitation_sum) as total_precipitation_mm
FROM `db_hackathon_2-28`.`default`.`county_weather_forecasts_16day`
WHERE forecast_date <= CURRENT_DATE() + INTERVAL 7 DAYS
GROUP BY state_name, county_name
ORDER BY total_precipitation_mm DESC
LIMIT 10;

-- Average temperature forecast by state
SELECT 
    state_name,
    AVG(temperature_2m_max) as avg_max_temp,
    AVG(temperature_2m_min) as avg_min_temp
FROM `db_hackathon_2-28`.`default`.`county_weather_forecasts_16day`
GROUP BY state_name
ORDER BY avg_max_temp DESC;
```

### Notes

* The Open-Meteo API is free but has usage limits (10,000 calls/day)
* Forecasts are updated when you re-run this notebook
* Geocoding uses OpenStreetMap's Nominatim service (rate-limited to 1 request/second)
* To update forecasts, simply re-run all cells

### Data Source

[Open-Meteo Weather API](https://open-meteo.com/) - Free weather forecast API